# Q1. Supervised Learning — Heart Disease Prediction

## 1. Data Loading and Inspection
Load `q1_heart_disease.csv` and display its shape, data types, and missing value counts. Show the first five rows.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score

# Load dataset
df = pd.read_csv('../data/q1_heart_disease.csv')

# Display basic information
print(f"Shape of the dataset: {df.shape}")
print("\nData Types:\n", df.dtypes)
print("\nMissing Value Counts:\n", df.isnull().sum())
df.head()

Shape of the dataset: (800, 12)

Data Types:
 age                  int64
sex                  int64
chest_pain_type     object
resting_bp         float64
cholesterol        float64
fasting_bs           int64
resting_ecg         object
max_hr               int64
exercise_angina      int64
oldpeak            float64
st_slope            object
heart_disease        int64
dtype: object

Missing Value Counts:
 age                 0
sex                 0
chest_pain_type     0
resting_bp         24
cholesterol        32
fasting_bs          0
resting_ecg         0
max_hr              0
exercise_angina     0
oldpeak             0
st_slope            0
heart_disease       0
dtype: int64


## 2. Exploratory Data Analysis
Produce at least three visualisations — include at minimum a target class distribution plot and a correlation heatmap. Follow each plot with a markdown cell interpreting what the chart reveals about the data.

In [ ]:
# 1. Target Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='heart_disease', data=df, palette='viridis')
plt.title('Distribution of Heart Disease (Target)')
plt.show()

**Interpretation:** The dataset is relatively balanced between patients with and without heart disease, which is ideal for training classification models without significant bias towards a majority class.

In [ ]:
# 2. Correlation Heatmap
plt.figure(figsize=(12, 8))
# Only numeric columns for correlation
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

**Interpretation:** The heatmap shows that `oldpeak`, `max_hr`, and `age` have notable correlations with the target variable. `max_hr` shows a negative correlation, suggesting that higher heart rates might be associated with lower heart disease presence in this specific dataset sample.

In [ ]:
# 3. Age distribution by Heart Disease
plt.figure(figsize=(8, 5))
sns.boxplot(x='heart_disease', y='age', data=df, palette='magma')
plt.title('Age Distribution by Heart Disease Status')
plt.show()

**Interpretation:** Patients with heart disease tend to be slightly older on average than those without, although there is significant overlap between the two groups.

## 3. Data Preprocessing
Handle missing values, apply one-hot encoding, scale numerical features, and split the data.

In [ ]:
# Identify column types
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.drop(columns=['heart_disease']).select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

# Preprocessing Pipelines
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numerical_cols),
        ('cat', cat_transformer, categorical_cols)
    ]
)

# Split data
X = df.drop('heart_disease', axis=1)
y = df['heart_disease']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

Categorical columns: ['chest_pain_type', 'resting_ecg', 'st_slope']
Numerical columns: ['age', 'sex', 'resting_bp', 'cholesterol', 'fasting_bs', 'max_hr', 'exercise_angina', 'oldpeak']
Train set size: (640, 11)
Test set size: (160, 11)


**Strategy:** Median imputation is used for numerical features to minimize the effect of outliers, while most-frequent imputation is used for categorical features. StandardScaler is essential for distance-based or gradient-based models, and one-hot encoding handles categorical variables without implying an incorrect ordinal relationship.

## 4 & 5. Model Training and Evaluation
Train Decision Tree, Random Forest, and Gradient Boosting. Evaluate using confusion matrix, precision, recall, and F1-score.

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    clf = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    results[name] = {
        'F1': f1_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'CM': confusion_matrix(y_test, y_pred)
    }
    
    print(f"--- {name} ---")
    print(f"Confusion Matrix:\n{results[name]['CM']}")
    print(classification_report(y_test, y_pred))
    print("\n")

--- Decision Tree ---
Confusion Matrix:
[[56 23]
 [22 59]]
              precision    recall  f1-score   support

           0       0.72      0.71      0.71        79
           1       0.72      0.73      0.72        81

    accuracy                           0.72       160
   macro avg       0.72      0.72      0.72       160
weighted avg       0.72      0.72      0.72       160



--- Random Forest ---
Confusion Matrix:
[[60 19]
 [15 66]]
              precision    recall  f1-score   support

           0       0.80      0.76      0.78        79
           1       0.78      0.81      0.80        81

    accuracy                           0.79       160
   macro avg       0.79      0.79      0.79       160
weighted avg       0.79      0.79      0.79       160



--- Gradient Boosting ---
Confusion Matrix:
[[61 18]
 [18 63]]
              precision    recall  f1-score   support

           0       0.77      0.77      0.77        79
           1       0.78      0.78      0.78        8

**Best Model Conclusion:** Based on the results, the **Random Forest** and **Gradient Boosting** models typically outperform the single Decision Tree due to their ensemble nature, which reduces variance and overfitting. I will proceed with Random Forest for hyperparameter tuning as it provides a strong balance between performance and interpretability (feature importance).

## 6. Hyperparameter Tuning
Tune at least one hyperparameter on the best-performing model using GridSearchCV.

In [ ]:
# Tuning Random Forest
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))])

param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(rf_pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation F1 Score: {grid_search.best_score_:.4f}")

# Compare with untuned baseline
untuned_f1 = results['Random Forest']['F1']
tuned_f1 = f1_score(y_test, grid_search.predict(X_test))

print(f"\nUntuned Test F1 Score: {untuned_f1:.4f}")
print(f"Tuned Test F1 Score: {tuned_f1:.4f}")

Best Parameters: {'classifier__max_depth': 10, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}
Best Cross-Validation F1 Score: 0.8270

Untuned Test F1 Score: 0.7952
Tuned Test F1 Score: 0.7976
